In [ ]:
!pip install -q langchain-huggingface langchain-qdrant langchain-community sentence-transformers rank_bm25 nltk numpy pandas matplotlib


In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder
from nltk.tokenize import word_tokenize

Setting configs

In [10]:
QDRANT_URL     = ""
QDRANT_API_KEY = ""

QDRANT_TOP = 10
BM25_TOP   = 10
MERGED_CAP = 20
RERANK_TOP = 5


EMBED_MODEL     = "abhinand/MedEmbed-base-v0.1"
COLLECTION_NAME = "ada_model_medembed_base_v0.1"
RERANKER_MODEL  = "BAAI/bge-reranker-base"

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:

EVAL_PATH = Path("/content/eval_queries_multi_chunk.json")
CHUNKS_PATH = Path("/content/drive/MyDrive/processed_chunks")

with open(EVAL_PATH) as f:
    eval_queries = json.load(f)

processed_chunks = []
for file_path in CHUNKS_PATH.glob("*.json"):
    with open(file_path, 'r') as f:
        processed_chunks.extend(json.load(f))

In [11]:
print(f"Loaded {len(eval_queries)} queries")
print(f"Loaded {len(processed_chunks)} chunks for BM25 index")

Loaded 30 queries
Loaded 2632 chunks for BM25 index


Metrics

In [12]:
def precision_at_k(retrieved: list, relevant: list, k: int = 5) -> float:
    return len(set(retrieved[:k]) & set(relevant)) / k

def recall_at_k(retrieved: list, relevant: list, k: int = 10) -> float:
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)

def mrr(retrieved: list, relevant: list) -> float:
    rel_set = set(relevant)
    for i, cid in enumerate(retrieved, 1):
        if cid in rel_set:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved: list, scores: dict, k: int = 5) -> float:
    dcg  = sum(scores.get(cid, 0) / np.log2(i + 1)
               for i, cid in enumerate(retrieved[:k], 1))
    idcg = sum(s / np.log2(i + 1)
               for i, s in enumerate(sorted(scores.values(), reverse=True)[:k], 1))
    return dcg / idcg if idcg > 0 else 0.0

def hit_at_k(retrieved: list, relevant: list, k: int = 5) -> float:
    return 1.0 if set(retrieved[:k]) & set(relevant) else 0.0

Building QDrant retriever and BM25 index

In [ ]:
# Load MedEmbed + connect to existing Qdrant collection
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = QdrantVectorStore.from_existing_collection(
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)
qdrant_retriever = vector_store.as_retriever(search_kwargs={"k": QDRANT_TOP})

In [15]:
# Build BM25 index from processed_chunks.json using word_tokenize
bm25_docs = [
    Document(
        page_content=chunk.get("page_content", ""),
        metadata={"chunk_id": chunk["metadata"]["chunk_id"]}
    )
    for i, chunk in enumerate(processed_chunks)
]
bm25_retriever = BM25Retriever.from_documents(
    bm25_docs,
    k=BM25_TOP,
    preprocess_func=word_tokenize,
)

print(f"BM25 index built over {len(bm25_docs)} chunks (top-{BM25_TOP})")


BM25 index built over 2632 chunks (top-10)


In [ ]:
#Loading the reranker

cross_encoder = CrossEncoder(RERANKER_MODEL)

Hybrid Eval Loop

In [17]:
def evaluate_hybrid() -> dict:
    """Run MedEmbed + BM25 → merge+deduplicate → bge-reranker-base → top-5."""
    p5_list, r10_list, mrr_list, ndcg5_list, hit5_list, lat_list = [], [], [], [], [], []

    for i, item in enumerate(eval_queries, 1):
        query        = item["query"]
        relevant_ids = item["ground_truth"]["relevant_chunk_ids"]
        rel_scores   = item["ground_truth"]["relevance_scores"]

        t0 = time.time()

        # Retrieve top-20 from Qdrant (semantic)
        qdrant_docs = qdrant_retriever.invoke(query)

        # Retrieve top-20 from BM25 (keyword)
        bm25_docs_q = bm25_retriever.invoke(query)

        # Merge and deduplicate by page_content, preserving order (Qdrant first)
        seen, merged = set(), []
        for doc in qdrant_docs + bm25_docs_q:
            key = doc.page_content[:200]
            if key not in seen:
                seen.add(key)
                merged.append(doc)

        # Rerank merged candidates with bge-reranker-base → top-5
        passages  = [doc.page_content for doc in merged]
        ce_inputs = [[query, p] for p in passages]
        ce_scores = cross_encoder.predict(ce_inputs)

        ranked_docs  = sorted(zip(ce_scores, merged), key=lambda x: x[0], reverse=True)
        top_docs     = [doc for _, doc in ranked_docs[:RERANK_TOP]]
        lat          = time.time() - t0

        # Extract chunk IDs for metric computation
        retrieved_ids = [doc.metadata.get("chunk_id") for doc in top_docs]
        all_reranked  = [doc.metadata.get("chunk_id") for _, doc in ranked_docs]

        p5    = precision_at_k(retrieved_ids, relevant_ids, k=5)
        r10   = recall_at_k(all_reranked,     relevant_ids, k=10)
        mrr_s = mrr(retrieved_ids,             relevant_ids)
        ndcg5 = ndcg_at_k(retrieved_ids,       rel_scores,  k=5)
        hit5  = hit_at_k(retrieved_ids,         relevant_ids, k=5)

        p5_list.append(p5);  r10_list.append(r10);  mrr_list.append(mrr_s)
        ndcg5_list.append(ndcg5);  hit5_list.append(hit5);  lat_list.append(lat)

        print(f"  [{i:02d}] P@5={p5:.2f} NDCG@5={ndcg5:.2f} Hit@5={int(hit5)} "
              f"candidates={len(merged):02d} | {lat*1000:.0f}ms | {query[:50]}…")

    return {
        "Pipeline":    "MedEmbed + BM25 + bge-reranker-base",
        "Precision@5": round(float(np.mean(p5_list)),   3),
        "Recall@10":   round(float(np.mean(r10_list)),  3),
        "MRR":         round(float(np.mean(mrr_list)),  3),
        "NDCG@5":      round(float(np.mean(ndcg5_list)),3),
        "Hit@5":       round(float(np.mean(hit5_list)), 3),
        "Latency_ms":  round(float(np.mean(lat_list)) * 1000, 1),
    }

exp3_result = evaluate_hybrid()
print(f"\n  Precision@5 : {exp3_result['Precision@5']}")
print(f"  Recall@10   : {exp3_result['Recall@10']}")
print(f"  MRR         : {exp3_result['MRR']}")
print(f"  NDCG@5      : {exp3_result['NDCG@5']}")
print(f"  Hit@5       : {exp3_result['Hit@5']}")
print(f"  Latency     : {exp3_result['Latency_ms']} ms")

  [01] P@5=0.60 NDCG@5=0.98 Hit@5=1 candidates=14 | 1094ms | A 55-year-old patient with advanced breast cancer …
  [02] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=17 | 540ms | What patient education should be provided to an in…
  [03] P@5=0.40 NDCG@5=0.29 Hit@5=1 candidates=15 | 571ms | A 30-year-old patient with type 1 diabetes, 20 wee…
  [04] P@5=0.80 NDCG@5=0.90 Hit@5=1 candidates=18 | 582ms | A 45-year-old patient with a history of type 2 dia…
  [05] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=18 | 511ms | An 80-year-old patient with diabetes is being cons…
  [06] P@5=0.00 NDCG@5=0.00 Hit@5=0 candidates=16 | 460ms | A 55-year-old patient with type 2 diabetes and sta…
  [07] P@5=0.40 NDCG@5=0.60 Hit@5=1 candidates=18 | 457ms | A 45-year-old patient with type 2 diabetes and obe…
  [08] P@5=0.60 NDCG@5=0.57 Hit@5=1 candidates=15 | 503ms | A 65-year-old patient with diabetes is having trou…
  [09] P@5=0.00 NDCG@5=0.00 Hit@5=0 candidates=19 | 526ms | A 40-year-old patient with type 1 diabetes 

In [19]:
exp1_baseline = {
    "Pipeline":    "MedEmbed only (no reranker)",
    "Precision@5": 0.393,
    "Recall@10":   0.562,
    "MRR":         0.797,
    "NDCG@5":      0.511,
    "Hit@5":       0.867,
    "Latency_ms":  249.8,
}
exp2_baseline = {
    "Pipeline":    "MedEmbed + bge-reranker-base",
    "Precision@5": 0.407,   # ← fill in from reranker_evaluation_results.csv
    "Recall@10":   0.58,
    "MRR":         0.765,
    "NDCG@5":      0.493,
    "Hit@5":       0.9,
    "Latency_ms":  821.0,
}

df = pd.DataFrame([exp1_baseline, exp2_baseline, exp3_result])

print("\n" + "="*80)
print("HYBRID RETRIEVAL EVALUATION — Experiments 1 / 2 / 3")
print(f"Retrieve K={RETRIEVE_K} per source → merge → bge-reranker-base → top-{RERANK_TOP}")
print("="*80)
print(df.to_string(index=False))



HYBRID RETRIEVAL EVALUATION — Experiments 1 / 2 / 3
Retrieve K=10 per source → merge → bge-reranker-base → top-5
                           Pipeline  Precision@5  Recall@10   MRR  NDCG@5  Hit@5  Latency_ms
        MedEmbed only (no reranker)        0.393      0.562 0.797   0.511  0.867       249.8
       MedEmbed + bge-reranker-base        0.407      0.580 0.765   0.493  0.900       821.0
MedEmbed + BM25 + bge-reranker-base        0.393      0.590 0.722   0.470  0.867       506.1


Adding BM25 increased Recall@10 from 0.580 to 0.590, indicating improved retrieval coverage. However, Precision@5 and ranking metrics decreased slightly, suggesting that MedEmbed embeddings already provide highly relevant semantic candidates. BM25 introduced additional keyword-based candidates but also added noise, reducing reranking effectiveness. Therefore, MedEmbed with cross-encoder reranking was selected as the optimal configuration.

## Increasing k caps to 15 per retriever -> 30 max

In [21]:
QDRANT_TOP = 15
BM25_TOP   = 15
MERGED_CAP = 30

In [22]:
qdrant_retriever = vector_store.as_retriever(search_kwargs={"k": QDRANT_TOP})
bm25_retriever = BM25Retriever.from_documents(
    bm25_docs,
    k=BM25_TOP,
    preprocess_func=word_tokenize,
)

In [27]:
exp3_result = evaluate_hybrid()
print(f"\n  Precision@5 : {exp3_result['Precision@5']}")
print(f"  Recall@10   : {exp3_result['Recall@10']}")
print(f"  MRR         : {exp3_result['MRR']}")
print(f"  NDCG@5      : {exp3_result['NDCG@5']}")
print(f"  Hit@5       : {exp3_result['Hit@5']}")
print(f"  Latency     : {exp3_result['Latency_ms']} ms")

  [01] P@5=0.60 NDCG@5=0.98 Hit@5=1 candidates=24 | 967ms | A 55-year-old patient with advanced breast cancer …
  [02] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=25 | 744ms | What patient education should be provided to an in…
  [03] P@5=0.40 NDCG@5=0.29 Hit@5=1 candidates=24 | 763ms | A 30-year-old patient with type 1 diabetes, 20 wee…
  [04] P@5=0.80 NDCG@5=0.90 Hit@5=1 candidates=28 | 828ms | A 45-year-old patient with a history of type 2 dia…
  [05] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=25 | 708ms | An 80-year-old patient with diabetes is being cons…
  [06] P@5=0.00 NDCG@5=0.00 Hit@5=0 candidates=22 | 721ms | A 55-year-old patient with type 2 diabetes and sta…
  [07] P@5=0.40 NDCG@5=0.60 Hit@5=1 candidates=25 | 665ms | A 45-year-old patient with type 2 diabetes and obe…
  [08] P@5=0.80 NDCG@5=0.72 Hit@5=1 candidates=21 | 747ms | A 65-year-old patient with diabetes is having trou…
  [09] P@5=0.20 NDCG@5=0.18 Hit@5=1 candidates=29 | 949ms | A 40-year-old patient with type 1 diabetes h

In [30]:
exp1_baseline = {
    "Pipeline":    "MedEmbed only (no reranker)",
    "Precision@5": 0.393,
    "Recall@10":   0.562,
    "MRR":         0.797,
    "NDCG@5":      0.511,
    "Hit@5":       0.867,
    "Latency_ms":  249.8,
}
exp2_baseline = {
    "Pipeline":    "MedEmbed + bge-reranker-base",
    "Precision@5": 0.407,
    "Recall@10":   0.58,
    "MRR":         0.765,
    "NDCG@5":      0.493,
    "Hit@5":       0.9,
    "Latency_ms":  821.0,
}
exp3_baseline = {
    "Pipeline":    "MedEmbed + BM25 + bge-reranker-base (k=10)",
    "Precision@5": 0.393,
    "Recall@10":   0.590,
    "MRR":         0.722,
    "NDCG@5":      0.470,
    "Hit@5":       0.867,
    "Latency_ms":  506.1
}



df = pd.DataFrame([exp1_baseline, exp2_baseline, exp3_baseline, exp3_result])

print("\n" + "="*80)
print("HYBRID RETRIEVAL EVALUATION")
print("="*80)
print(df.to_string(index=False))


HYBRID RETRIEVAL EVALUATION
                                  Pipeline  Precision@5  Recall@10   MRR  NDCG@5  Hit@5  Latency_ms
               MedEmbed only (no reranker)        0.393      0.562 0.797   0.511  0.867       249.8
              MedEmbed + bge-reranker-base        0.407      0.580 0.765   0.493  0.900       821.0
MedEmbed + BM25 + bge-reranker-base (k=10)        0.393      0.590 0.722   0.470  0.867       506.1
MedEmbed + BM25 + bge-reranker-base (k=15)        0.407      0.583 0.731   0.484  0.900       783.2


## Increase k cap to 20 per source , MAX = 40

In [31]:
QDRANT_TOP = 20
BM25_TOP   = 20
MERGED_CAP = 40

In [32]:
qdrant_retriever = vector_store.as_retriever(search_kwargs={"k": QDRANT_TOP})
bm25_retriever = BM25Retriever.from_documents(
    bm25_docs,
    k=BM25_TOP,
    preprocess_func=word_tokenize,
)

In [33]:
exp5_result = evaluate_hybrid()

  [01] P@5=0.60 NDCG@5=0.98 Hit@5=1 candidates=33 | 1305ms | A 55-year-old patient with advanced breast cancer …
  [02] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=35 | 1073ms | What patient education should be provided to an in…
  [03] P@5=0.40 NDCG@5=0.29 Hit@5=1 candidates=31 | 1021ms | A 30-year-old patient with type 1 diabetes, 20 wee…
  [04] P@5=0.80 NDCG@5=0.90 Hit@5=1 candidates=36 | 1025ms | A 45-year-old patient with a history of type 2 dia…
  [05] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=33 | 867ms | An 80-year-old patient with diabetes is being cons…
  [06] P@5=0.20 NDCG@5=0.18 Hit@5=1 candidates=28 | 866ms | A 55-year-old patient with type 2 diabetes and sta…
  [07] P@5=0.40 NDCG@5=0.60 Hit@5=1 candidates=35 | 958ms | A 45-year-old patient with type 2 diabetes and obe…
  [08] P@5=0.60 NDCG@5=0.57 Hit@5=1 candidates=29 | 932ms | A 65-year-old patient with diabetes is having trou…
  [09] P@5=0.20 NDCG@5=0.18 Hit@5=1 candidates=37 | 1072ms | A 40-year-old patient with type 1 diabe

In [34]:
exp5_result_updated = exp5_result.copy()
exp5_result_updated["Pipeline"] = "MedEmbed + BM25 + bge-reranker-base (k=20)"

df = pd.DataFrame([exp1_baseline, exp2_baseline, exp3_baseline, exp3_result, exp5_result_updated])

print("\n" + "="*80)
print("HYBRID RETRIEVAL EVALUATION")
print("="*80)
print(df.to_string(index=False))


HYBRID RETRIEVAL EVALUATION
                                  Pipeline  Precision@5  Recall@10   MRR  NDCG@5  Hit@5  Latency_ms
               MedEmbed only (no reranker)        0.393      0.562 0.797   0.511  0.867       249.8
              MedEmbed + bge-reranker-base        0.407      0.580 0.765   0.493  0.900       821.0
MedEmbed + BM25 + bge-reranker-base (k=10)        0.393      0.590 0.722   0.470  0.867       506.1
MedEmbed + BM25 + bge-reranker-base (k=15)        0.407      0.583 0.731   0.484  0.900       783.2
MedEmbed + BM25 + bge-reranker-base (k=20)        0.420      0.594 0.747   0.495  0.933       995.9


Interpretation: Hybrid retrieval (bm25 + vector) performance benefits with candidate pool size but latency also increases

Increasing the hybrid candidate pool size from k=10 to k=20 improved Precision@5 (0.393→0.420) and Hit@5 (0.867→0.933), indicating that the cross-encoder reranker benefits from larger retrieval pools. However, latency nearly doubled, highlighting a trade-off between ranking quality and inference cost.

In [36]:
df.to_csv('hybrid_retrieval_evaluation.csv', index=False)
print('DataFrame saved to hybrid_retrieval_evaluation.csv')

DataFrame saved to hybrid_retrieval_evaluation.csv
